# 1a – Hybrid RAG

Same pipeline as Example 1. Load, chunk and embed stay as they are, and Chroma still holds the vectors. Step 4 adds a second index, BM25, over those same chunks. Step 5 searches both indexes and fuses the two lists.

| List | What it scores | What it is for |
|---|---|---|
| **Lexical: BM25** | exact words, weighted by how rare the word is | product codes, names, clause numbers, a version such as `7.2` |
| **Dense: embeddings** | meaning | a paraphrase. "How long may a loan last" when the text says "the maximum term is 84 months" |

BM25 is the list to trust for an identifier. The embedding model is the list to trust when the question and the answer share a meaning and few words. On a contract corpus the same split is "IFRS 16" against "the lease accounting rule", and "terminate with six months' notice" against "notice period".

A BM25 score of 5 and a cosine similarity of 0.5 live on different scales. Reciprocal rank fusion (Cormack, Clarke and Buettcher, SIGIR 2009) uses the ranks alone, so the scales never have to be lined up:

```
score(d) = Σ  1 / (k + rank_r(d))     over both rankings, k = 60
```

Rank 1 in one list contributes 1/61 ≈ 0.0164. Rank 1 in both lists contributes twice that. A chunk that one list never returned contributes nothing from that list.

With `k = 60` a short list is nearly flat: rank 4 still contributes 1/64 ≈ 0.0156. What moves the fused score is appearing in both lists. A chunk both lists rank well beats a chunk that only one list loves.

Anthropic (2024), [Contextual Retrieval](https://www.anthropic.com/research/contextual-retrieval), measured this at a larger scale. Embedding search left the right chunk out of the top 20 for 3.7% of questions. Fusing that list with a BM25 list cut the misses to 2.9%, on the same chunks. (Both of those indexes used Anthropic's contextualized chunks; the gain on top of that is the second, lexical list.)

**Before you run this notebook**

- Same setup as Example 1: dependencies from `requirements.txt`, Jupyter started from this folder, Ollama running with `mistral-small3.2:24b` and `embeddinggemma:300m`.
- BM25 is a short function in this notebook, using the standard library. The models are the two from Example 1.


## Imports

The same LangChain pieces as Example 1. From the standard library: `math` for the BM25 weights, `re` for the tokenizer, `Counter` for term frequencies.


In [1]:
import math
import os
import re
from collections import Counter
from pathlib import Path

from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


## Configuration

Shared with Example 1: `DATA_DIR`, `LLM_MODEL`, `EMBEDDING_MODEL`, `CHUNK_SIZE`, `CHUNK_OVERLAP`, `TOP_K`.

New:

- `BM25_K1` – how fast repeated words stop helping. `1.5` is the usual Okapi value: the second occurrence counts, the tenth barely more than the ninth.
- `BM25_B` – how strongly the chunk length is normalized. `0.75` is the usual value. A long chunk has more chances to contain a query word, and this pulls that advantage back.
- `RRF_K` – the constant in `1 / (k + rank)`. `60` is the value from Cormack et al. (2009).
- `CANDIDATE_K` – how many hits each list hands to the fusion. Larger than `TOP_K`: a chunk only has to sit somewhere in a list to earn a score. On a real corpus this is often 20–100; here the whole corpus is 19 chunks, so 10 is half of it.


In [2]:
DATA_DIR = Path("data")

LLM_MODEL = os.getenv("RAG_LLM", "ollama:mistral-small3.2:24b")

EMBEDDING_MODEL = "embeddinggemma:300m"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
TOP_K = 4

BM25_K1 = 1.5
BM25_B = 0.75
RRF_K = 60
CANDIDATE_K = 10

print("LLM:", LLM_MODEL)


LLM: ollama:mistral-small3.2:24b


## Step 1 – LOAD

As in Example 1: every `.txt` file in `data/` becomes one `Document`, with the file name in `metadata["source"]`.


In [3]:
docs = [
    Document(page_content=path.read_text(encoding="utf-8"),
             metadata={"source": path.name})
    for path in sorted(DATA_DIR.glob("*.txt"))
]
print(f"Loaded {len(docs)} documents")
for doc in docs:
    print(f"  {doc.metadata['source']}: {len(doc.page_content)} characters")


Loaded 3 documents
  harbor_city_transit.txt: 1624 characters
  meridian_credit_policy.txt: 2110 characters
  novaterra_annual_report.txt: 2236 characters


## Step 2 – CHUNK

Same splitter as Example 1. It tries the largest natural boundary first: paragraph, line, sentence, word, character. `add_start_index=True` records where the chunk sat in its document.

`chunk_id` is the chunk's position in this list. Both indexes store it, and the fusion uses it to tell that a BM25 hit and a Chroma hit are the same chunk.


In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", r"(?<=\. )", " ", ""],
    is_separator_regex=True,
    add_start_index=True,
)
chunks = splitter.split_documents(docs)
for chunk_id, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = chunk_id
print(f"Split into {len(chunks)} chunks")


Split into 19 chunks


The chunk that states Novaterra's revenue. It says "the company" and never the name. The lexical search below has nothing to match "Novaterra" against in this chunk; the dense search still ranks it first for that question.


In [5]:
chunk = next(c for c in chunks if "revenue rose" in c.page_content)
print(chunk.metadata)
print(chunk.page_content)


{'source': 'novaterra_annual_report.txt', 'start_index': 704, 'chunk_id': 14}
Financial Performance

The company's revenue rose to EUR 4.2 billion in 2025, an increase of 9 percent over the prior year. Growth was driven mainly by higher output from its onshore wind portfolio and by new long-term supply contracts with industrial customers. EBITDA reached EUR 910 million, while net income came in at EUR 312 million. The board proposes a dividend of EUR 1.40 per share, up from EUR 1.25.


## Step 3 – EMBED

Same local embedding model as Example 1. Ollama returns unit-length vectors, so the inner product is the cosine similarity. One sentence, to see the shape: 768 floats.


In [6]:
embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)

vector = embeddings.embed_query("What was the revenue in 2025?")
print(len(vector), "dimensions, first five values:", [round(v, 3) for v in vector[:5]])


768 dimensions, first five values: [-0.077, 0.018, 0.024, 0.036, -0.012]


## Step 4 – STORE

Two indexes over the same chunks.

**Dense.** Chroma, as in Example 1, cosine distance, collection emptied first so re-running the cell does not duplicate rows.

**Lexical.** A BM25 index in memory: for every chunk, how often each word occurs, plus an IDF weight per word. There is nothing to persist; rebuilding it is a pass over 19 short texts.

A word `t` that occurs in `n(t)` of the `N` chunks has

```
IDF(t) = ln( 1 + (N - n(t) + 0.5) / (n(t) + 0.5) )
```

A word in every chunk lands near zero. A word in one chunk lands above 2. The score of chunk `d` for one query word is

```
IDF(t) · tf · (k1 + 1) / (tf + k1 · (1 - b + b · |d| / avgdl))
```

`tf` is how often the word occurs in the chunk, `|d|` is the chunk length in tokens, `avgdl` is the average chunk length. The chunk's score is the sum of that over the query words. Each occurrence of a query word adds, so a repeated word counts again, while `k1` stops a chunk from winning by repeating one word forever.

The tokenizer lowercases and keeps a decimal number whole. The token for the policy version is `7.2`, and it occurs in one chunk. The digit `7` occurs in several, including the heading of section 7.


In [7]:
vector_store = Chroma(
    collection_name="hybrid_rag",
    embedding_function=embeddings,
    collection_metadata={"hnsw:space": "cosine"},
)
vector_store.reset_collection()
ids = vector_store.add_documents(chunks)
print(f"Indexed {len(ids)} chunks in Chroma")


Indexed 19 chunks in Chroma


In [8]:
def tokenize(text):
    return re.findall(r"\d+(?:\.\d+)+|[a-z0-9]+", text.lower())


def build_bm25(documents, k1=BM25_K1, b=BM25_B):
    tokenized = [tokenize(document.page_content) for document in documents]
    doc_count = len(tokenized)
    avgdl = sum(len(tokens) for tokens in tokenized) / doc_count
    term_freqs = []
    doc_freq = Counter()
    for tokens in tokenized:
        tf = Counter(tokens)
        term_freqs.append(tf)
        doc_freq.update(tf.keys())
    idf = {
        term: math.log((doc_count - freq + 0.5) / (freq + 0.5) + 1)
        for term, freq in doc_freq.items()
    }
    return {"term_freqs": term_freqs, "idf": idf, "avgdl": avgdl, "k1": k1, "b": b}


bm25_index = build_bm25(chunks)
print(
    f"BM25 index: {len(chunks)} chunks, {len(bm25_index['idf'])} terms, "
    f"average {bm25_index['avgdl']:.0f} tokens"
)
for term in ["the", "revenue", "kessel", "7.2"]:
    df = sum(term in tf for tf in bm25_index["term_freqs"])
    print(f"  {term:10}  idf={bm25_index['idf'][term]:.3f}  chunks={df}")


BM25 index: 19 chunks, 460 terms, average 53 tokens
  the         idf=0.025  chunks=19
  revenue     idf=1.743  chunks=3
  kessel      idf=2.079  chunks=2
  7.2         idf=2.590  chunks=1


## Step 5a – RETRIEVE

`bm25_search` scores every chunk and returns the ones above zero, best first, cut at `CANDIDATE_K`. A chunk that shares no query word is left off the list: a zero would still be a rank, and a rank is a vote.

`hybrid_search` asks Chroma for the same number of neighbours, then fuses the two id sequences. `hybrid_retrieve` keeps the top `TOP_K` fused chunks. That is the retriever the answer chain calls.

Fusion reads `RRF_K` and `CANDIDATE_K` when it runs, so the exercises below can change them and re-run the demo cell.


In [9]:
def bm25_search(index, question, k):
    scores = []
    for tf in index["term_freqs"]:
        length = sum(tf.values())
        score = 0.0
        norm = 1 - index["b"] + index["b"] * length / index["avgdl"]
        for term in tokenize(question):
            freq = tf.get(term, 0)
            if freq == 0:
                continue
            score += (
                index["idf"][term]
                * (freq * (index["k1"] + 1))
                / (freq + index["k1"] * norm)
            )
        scores.append(score)
    order = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    hits = [i for i in order if scores[i] > 0][:k]
    return hits, scores


def reciprocal_rank_fusion(rankings):
    fused = {}
    for ranking in rankings:
        for rank, chunk_id in enumerate(ranking, start=1):
            chunk_id = int(chunk_id)
            fused[chunk_id] = fused.get(chunk_id, 0.0) + 1.0 / (RRF_K + rank)
    order = sorted(fused, key=lambda chunk_id: (-fused[chunk_id], chunk_id))
    return order, fused


def hybrid_search(question):
    lexical_ids, lexical_scores = bm25_search(bm25_index, question, CANDIDATE_K)
    dense_hits = vector_store.similarity_search_with_relevance_scores(
        question, k=CANDIDATE_K
    )
    dense_ids = [int(doc.metadata["chunk_id"]) for doc, _score in dense_hits]
    dense_scores = {
        int(doc.metadata["chunk_id"]): score for doc, score in dense_hits
    }
    fused_ids, fused_scores = reciprocal_rank_fusion([lexical_ids, dense_ids])
    return {
        "lexical_ids": lexical_ids,
        "lexical_scores": lexical_scores,
        "dense_ids": dense_ids,
        "dense_scores": dense_scores,
        "fused_ids": fused_ids,
        "fused_scores": fused_scores,
    }


def hybrid_retrieve(question):
    found = hybrid_search(question)
    return [chunks[chunk_id] for chunk_id in found["fused_ids"][:TOP_K]]


def show_retrieval(question):
    found = hybrid_search(question)
    lexical_ids = found["lexical_ids"]
    lexical_scores = found["lexical_scores"]
    dense_ids = found["dense_ids"]
    dense_scores = found["dense_scores"]
    fused_ids = found["fused_ids"]
    fused_scores = found["fused_scores"]
    print(f"\n{question}")

    def print_list(title, ids, score_of):
        print(f"  {title}")
        if not ids:
            print("    (no hit)")
            return
        for rank, chunk_id in enumerate(ids[:TOP_K], start=1):
            chunk = chunks[chunk_id]
            source = Path(chunk.metadata["source"]).name
            text = chunk.page_content[:80].replace("\n", " ")
            print(f"    {rank}. {score_of(chunk_id):8.4f}  [{source}]  {text}")

    print_list("Lexical (BM25)", lexical_ids, lambda chunk_id: lexical_scores[chunk_id])
    print_list("Dense (embeddings)", dense_ids, lambda chunk_id: dense_scores[chunk_id])
    print_list("Fused", fused_ids, lambda chunk_id: fused_scores[chunk_id])


Three questions, chosen so the lists disagree. The fused block is what the LLM will see.

1. **`7.2`** — one chunk in the corpus contains that token: the header "Meridian Bank - Retail Credit Policy, Version 7.2". The embedding of the bare number sits next to the heading "7. Monitoring", and that section comes first in the dense list. Fusion follows BM25.
2. **The loan term, asked without those words** — the answer is "The maximum term is 84 months." The question below says "how long may an individual borrowing last", which shares its words with the eligibility rules ("at least", "the last two years") instead. BM25 opens on eligibility. The dense list opens on the loan-limits chunk. Fusion follows the embeddings.
3. **Novaterra's revenue** — BM25 opens on Harbor City's budget, which says "fare revenue" and "2025". The chunk with "EUR 4.2 billion" says "the company", so the name in the question never matches it. The dense list opens on that chunk anyway. Fusion follows the embeddings.


In [10]:
for question in [
    "7.2",
    "How long may an individual borrowing last at most?",
    "What was Novaterra's revenue in 2025?",
]:
    show_retrieval(question)



7.2
  Lexical (BM25)
    1.   2.4519  [meridian_credit_policy.txt]  Meridian Bank - Retail Credit Policy, Version 7.2 (Excerpt) (Fictional bank, cre
  Dense (embeddings)
    1.   0.2671  [meridian_credit_policy.txt]  7. Monitoring  The bank reviews the performance of the retail portfolio monthly.
    2.   0.2325  [meridian_credit_policy.txt]  Meridian Bank - Retail Credit Policy, Version 7.2 (Excerpt) (Fictional bank, cre
    3.   0.2218  [meridian_credit_policy.txt]  6. Overrides  A credit officer may override an automatic decline only with writt
    4.   0.1812  [meridian_credit_policy.txt]  2. Eligibility  An applicant must be at least 18 years old and must have been re
  Fused
    1.   0.0325  [meridian_credit_policy.txt]  Meridian Bank - Retail Credit Policy, Version 7.2 (Excerpt) (Fictional bank, cre
    2.   0.0164  [meridian_credit_policy.txt]  7. Monitoring  The bank reviews the performance of the retail portfolio monthly.
    3.   0.0159  [meridian_credit_policy.txt]  6. Ove

## Step 5b – GENERATE

The prompt is the one from Example 1: answer only from the context, say so when the answer is missing, cite the chunks as `[1]`, `[2]`.


In [11]:
PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You answer questions using ONLY the context below. "
     "If the context does not contain the answer, say you don't know. "
     "Cite the chunks you used like [1], [2].\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])


def format_docs(docs):
    return "\n\n".join(
        f"[{i}] (source: {Path(d.metadata['source']).name})\n{d.page_content}"
        for i, d in enumerate(docs, start=1)
    )


### Wiring it together with LCEL

`hybrid_retrieve` takes the place of `vector_store.as_retriever`. The rest of the chain is Example 1: fill the prompt, call the chat model at temperature 0, read the text off the message.

```
question ──┬─> hybrid_retrieve -> format_docs ──> {context}
           └─> RunnablePassthrough ─────────────> {question}
                                                      │
                               PROMPT -> LLM -> StrOutputParser
```


In [12]:
llm = init_chat_model(LLM_MODEL, temperature=0)

rag_chain = (
    {
        "context": RunnableLambda(hybrid_retrieve) | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | PROMPT
    | llm
    | StrOutputParser()
)


## Run it

For each question the three lists are printed, then the answer. The first question is the one BM25 gets wrong on its own. The second is the paraphrase.


In [13]:
QUESTIONS = [
    "What was Novaterra's revenue in 2025?",
    "How long may an individual borrowing last at most?",
    "Which credit score leads to an automatic decline at Meridian Bank?",
    "How often do the Harbor City ferries run in winter?",
]

for question in QUESTIONS:
    show_retrieval(question)
    print("\nAnswer:", rag_chain.invoke(question))
    print("-" * 80)



What was Novaterra's revenue in 2025?
  Lexical (BM25)
    1.   5.7731  [harbor_city_transit.txt]  Finances  The authority's operating budget for 2025 was 410 million dollars. Far
    2.   5.6840  [novaterra_annual_report.txt]  Financial Performance  The company's revenue rose to EUR 4.2 billion in 2025, an
    3.   5.3479  [novaterra_annual_report.txt]  Novaterra Energy AG - Annual Report 2025 (Excerpt) (Fictional company, created f
    4.   3.3425  [novaterra_annual_report.txt]  2025 was a year of transition. The company completed the sale of its last two co
  Dense (embeddings)
    1.   0.5467  [novaterra_annual_report.txt]  Financial Performance  The company's revenue rose to EUR 4.2 billion in 2025, an
    2.   0.5043  [novaterra_annual_report.txt]  Novaterra Energy AG - Annual Report 2025 (Excerpt) (Fictional company, created f
    3.   0.4580  [novaterra_annual_report.txt]  2025 was a year of transition. The company completed the sale of its last two co
    4.   0.4342  [novate

## Things to try

1. `show_retrieval("97.2")`, then `show_retrieval("97.2 percent")`. `97.2` is the availability of the Kessel Ridge wind farm, and it is a single lexical hit, so fusion follows BM25. Adding the word "percent" also matches the bank's monitoring section ("3 percent") and the punctuality paragraph. The embedding ranks those first, both lists now agree on them, and Kessel Ridge drops down the fused list. An identifier wins when it is the only lexical hit. A common companion word spends that advantage.
2. Set `CANDIDATE_K = 3` and re-run the demo cell. The loan-limits chunk is BM25's 4th hit, so it leaves the lexical list. Its fused score falls to the dense contribution alone, `1/61`, the same number as the eligibility chunk's lexical contribution. The tie breaks toward the smaller `chunk_id`, and eligibility stays on top.
3. Set `RRF_K = 1`, then `RRF_K = 1000`, and re-run the demo. A small `k` makes rank 1 much louder than rank 4 (`1/2` against `1/5`). A large `k` flattens the list further, so only "in both lists, or in one" still matters.
4. Ask your own question in the cell below. "What is the maximum term of a personal loan?" uses the words of the chunk; compare it with the paraphrase in the demo.


In [14]:
print(rag_chain.invoke("What is the maximum term of a personal loan?"))


The maximum term for an unsecured personal loan is 84 months [1].
